In [ ]:
!pwd

/content


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip cache purge

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install chemprop --no-deps
!pip install rdkit-pypi pandas pyarrow scikit-learn

In [ ]:

!pip install rdkit

In [ ]:
import pandas as pd

df = pd.read_parquet("dataset_raw.parquet")

print("\n=== RAW ===")
print("rows:", len(df))
print("unique smiles:", df["smiles"].nunique())
print("tasks:", df["task"].nunique())

print("\nSPLIT:")
print(df["split"].value_counts())

print("\nLABEL:")
print(df["label"].value_counts())



df["split"] = df["split"].replace({"valid": "val"})

print("\nSPLIT AFTER FIX:")
print(df["split"].value_counts())


df_wide = df.pivot_table(
    index="smiles",
    columns="task",
    values="label",
    aggfunc="first"
)

split_df = df.groupby("smiles")["split"].agg(lambda x: x.value_counts().index[0])

df_wide["split"] = split_df
df_wide = df_wide.reset_index()

print("\n=== WIDE ===")
print("rows:", len(df_wide))
print("cols:", len(df_wide.columns))
print(df_wide["split"].value_counts())




df_wide.to_csv("chemprop_data.csv", index=False)

tasks = [c for c in df_wide.columns if c not in ["smiles","split"]]

print("tasks:", len(tasks))





=== RAW ===
rows: 65127
unique smiles: 40000
tasks: 13

SPLIT:
split
train    45617
test     13002
valid     6508
Name: count, dtype: int64

LABEL:
label
0.0    38724
1.0    26403
Name: count, dtype: int64

SPLIT AFTER FIX:
split
train    45617
test     13002
val       6508
Name: count, dtype: int64

=== WIDE ===
rows: 40000
cols: 15
split
train    28989
test      7416
val       3595
Name: count, dtype: int64
tasks: 13


In [ ]:
targets = " ".join([f'"{t}"' for t in tasks])

!chemprop train \
  --data-path chemprop_data.csv \
  --task-type classification \
  --smiles-columns smiles \
  --target-columns {targets} \
  --splits-column split \
  --epochs 10 \
  --batch-size 64 \
  --message-hidden-dim 300 \
  --depth 3 \
  --output-dir chemprop_model

Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
Epoch 7/9  ━━━━━━━━━━━━━╺━━ 374/453 0:00:03 •        99.31it/s v_num: 0.000     
                                    0:00:01                    train_loss_step: 
                                                               0.489 val_loss:  
                                                               0.456            
                                                               train_loss_epoch:
Epoch 7/9  ━━━━━━━━━━━━━╺━━ 375/453 0:00:03 •        99.31it/s v_num: 0.000     
                                    0:00:01                    train_loss_step: 
                                                               0.451 val_loss:  
                                                               0.456            
                                                               train_loss_epoch:
Epoch 7/9  ━━━━━━━━━━━━━╺━━ 376/453 0:00:03 •        99.31it/s v_num: 0.000     
                                    0:00:01 

In [ ]:
!ls chemprop_model/model_0/checkpoints

'best-epoch=9-val_loss=0.45.ckpt'   last.ckpt


In [ ]:
!chemprop fingerprint \
  --test-path chemprop_data.csv \
  --model-path chemprop_model/model_0/checkpoints/best-epoch=9-val_loss=0.45.ckpt \
  --ffn-block-index -1 \
  --output embeddings.csv

2026-03-25T12:08:53 - INFO:chemprop.cli.main - Running in mode 'fingerprint' with args: {'smiles_columns': None, 'reaction_columns': None, 'no_header_row': False, 'num_workers': 0, 'batch_size': 64, 'accelerator': 'auto', 'devices': 'auto', 'rxn_mode': 'REAC_DIFF', 'multi_hot_atom_featurizer_mode': 'V2', 'keep_h': False, 'add_h': False, 'ignore_stereo': False, 'reorder_atoms': False, 'molecule_featurizers': None, 'descriptors_path': None, 'descriptors_columns': None, 'no_descriptor_scaling': False, 'no_atom_feature_scaling': False, 'no_atom_descriptor_scaling': False, 'no_bond_feature_scaling': False, 'no_bond_descriptor_scaling': False, 'atom_features_path': None, 'atom_descriptors_path': None, 'bond_features_path': None, 'bond_descriptors_path': None, 'constraints_path': None, 'constraints_to_targets': None, 'use_cuikmolmaker_featurization': False, 'test_path': PosixPath('chemprop_data.csv'), 'output': PosixPath('embeddings.csv'), 'model_paths': [PosixPath('chemprop_model/model_0/che

In [ ]:
import pandas as pd

emb = pd.read_csv("embeddings_0.csv")

smiles = pd.read_csv("chemprop_data.csv")["smiles"]

emb["smiles"] = smiles

print(emb.head())


print("shape:", emb.shape)
print("dim:", emb.shape[1]-1)
print("unique smiles:", emb["smiles"].nunique())



df = pd.read_parquet("dataset_raw.parquet")
df["split"] = df["split"].replace({"valid":"val"})

df = df.merge(emb, on="smiles")

print("rows:", len(df))
print("cols:", len(df.columns))


print("unique smiles:", df["smiles"].nunique())
print("tasks:", df["task"].nunique())
print("splits:", df["split"].value_counts())




       fp_0      fp_1      fp_2      fp_3      fp_4      fp_5      fp_6  \
0  0.148434  0.129267 -0.154176 -0.142482 -2.086064  0.772732 -0.283858   
1 -0.088535  0.550594 -0.132664 -0.079431 -0.213598  0.380932 -0.209037   
2  0.330495  0.340831 -0.129806 -0.177310 -1.007545  1.305478 -0.320872   
3  0.475148  0.198172 -0.089248 -0.095767  0.179321  0.605028 -0.204041   
4  0.138735  0.328898 -0.132852 -0.139891 -1.169359  0.854134 -0.263555   

       fp_7      fp_8      fp_9  ...    fp_291    fp_292    fp_293    fp_294  \
0  0.521799  0.026034 -0.075188  ... -0.521993  0.025752  0.029312  0.796180   
1 -0.598028  0.802959  0.437972  ... -0.307025  0.797227  0.263283  0.117893   
2  0.512125  0.114906  0.123856  ... -0.607681  0.186340 -0.085536  0.968963   
3 -0.070924  0.225564  0.352041  ... -0.306981  0.135895  0.052943  0.450827   
4  0.556117  0.193212  0.163339  ... -0.487490  0.209414 -0.064204  0.653260   

     fp_295    fp_296    fp_297    fp_298    fp_299  \
0 -0.158252  

In [ ]:

import pandas as pd

# embeddings już masz jako emb
emb.to_parquet(
    "embeddings.parquet",
    engine="pyarrow",
    compression="snappy"
)

print("saved: embeddings.parquet")


df.to_parquet(
    "dataset_with_embeddings.parquet",
    engine="pyarrow",
    compression="snappy"
)

print("saved: dataset_with_embeddings.parquet")


emb_check = pd.read_parquet("embeddings.parquet")
df_check = pd.read_parquet("dataset_with_embeddings.parquet")

print("emb:", emb_check.shape)
print("df:", df_check.shape)


saved: embeddings.parquet
saved: dataset_with_embeddings.parquet
emb: (40000, 301)
df: (65127, 513)


In [ ]:
emb_cols = [c for c in df.columns if c.startswith("fp_")]

assert len(emb_cols) == 300
assert len(df) == 65127
assert df["task"].nunique() == 13
assert set(df["split"].unique()) == {"train","val","test"}

print("FINAL DATA READY ✅")

FINAL DATA READY ✅
